# 🛡️ Burglary Detection — EfficientNet-B0 Training (Keras) — Kaggle Version

**No dataset upload needed! UCF Crime dataset is added directly from Kaggle.**

### Setup Steps Before Running:
1. Click **+ Add Data** (right sidebar) → Search **'UCF Crime Dataset'**
2. Add the dataset with frames (look for one that has `Train/Burglary/` folders)
3. Go to **Settings** (right sidebar) → Enable **GPU Accelerator**
4. Run cells one by one from top to bottom

### After Training:
- Model saved to `/kaggle/working/action_classifier.keras`
- Download from the **Output** tab on the right sidebar
- Place it in your `d:\HOME-cctv\` folder on your PC

---
## Cell 1 — Check GPU

In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

if tf.config.list_physical_devices('GPU'):
    print('✅ GPU is active — training will be fast!')
else:
    print('⚠️ No GPU! Go to Settings (right panel) → Accelerator → GPU')

---
## Cell 2 — Explore Dataset Structure

Run this cell first to see what folders are in your added dataset.
This helps confirm the correct path before loading images.

In [ ]:
import os

# All Kaggle datasets are in /kaggle/input/
print('=== Kaggle Input Datasets ===')
for dataset_name in os.listdir('/kaggle/input/'):
    dataset_path = f'/kaggle/input/{dataset_name}'
    print(f'\nDataset: {dataset_name}/')
    # Show top-level folders inside dataset
    for item in sorted(os.listdir(dataset_path))[:15]:
        full_path = os.path.join(dataset_path, item)
        if os.path.isdir(full_path):
            sub_count = len(os.listdir(full_path))
            print(f'  {item}/ ({sub_count} items inside)')
        else:
            size_mb = os.path.getsize(full_path) / 1e6
            print(f'  {item} ({size_mb:.1f} MB)')

---
## Cell 3 — Set Dataset Path

⚠️ **IMPORTANT: Update `DATASET_NAME` based on Cell 2 output!**

Look at Cell 2 output and find the folder name that contains `Train/` folder.
Then set `DATASET_NAME` to that folder name below.

In [ ]:
# =========================================================
# UPDATE THIS LINE based on Cell 2 output!
# Example: if you see 'ucf-crime-dataset' in Cell 2, set:
# DATASET_NAME = 'ucf-crime-dataset'
# =========================================================
DATASET_NAME = 'ucf-crime-dataset'   # ← Change this if needed!

TRAIN_PATH = f'/kaggle/input/{DATASET_NAME}/Train'

# Verify the path exists
if os.path.exists(TRAIN_PATH):
    print(f'✅ Train folder found at: {TRAIN_PATH}')
    print(f'   Subfolders: {os.listdir(TRAIN_PATH)}')
else:
    print(f'❌ Path not found: {TRAIN_PATH}')
    print('   Check Cell 2 output and update DATASET_NAME above!')

---
## Cell 4 — Import Libraries

In [ ]:
import random
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.applications  import EfficientNetB0
from tensorflow.keras.layers        import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras               import Model, Input
from tensorflow.keras.optimizers    import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks     import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from PIL import Image
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print('✅ All libraries imported!')

---
## Cell 5 — Collect File Paths (No ZIP — Direct Folder Reading)

In [ ]:
from pathlib import Path

BURGLARY_PATH = os.path.join(TRAIN_PATH, 'Burglary')
NORMAL_PATH   = os.path.join(TRAIN_PATH, 'NormalVideos')

# Collect all image file paths from the folders
# (No ZIP needed — Kaggle gives direct folder access!)
burglary_files = sorted([str(p) for p in Path(BURGLARY_PATH).rglob('*.png')])
normal_files   = sorted([str(p) for p in Path(NORMAL_PATH).rglob('*.png')])

# Also check for .jpg in case dataset uses jpg
if len(burglary_files) == 0:
    burglary_files = sorted([str(p) for p in Path(BURGLARY_PATH).rglob('*.jpg')])
    normal_files   = sorted([str(p) for p in Path(NORMAL_PATH).rglob('*.jpg')])

print(f'Found in dataset:')
print(f'  Burglary frames : {len(burglary_files):,}')
print(f'  Normal frames   : {len(normal_files):,}')

if len(burglary_files) == 0:
    print('\n❌ No images found! Check DATASET_NAME in Cell 3 and folder structure.')

---
## Cell 6 — Balanced Sampling

In [ ]:
SUBSET_SIZE      = len(burglary_files)                          # Use ALL burglary
burglary_sampled = burglary_files                               # All burglary images
normal_sampled   = random.sample(normal_files, SUBSET_SIZE)     # Equal normal images

print(f'Balanced dataset:')
print(f'  Burglary (label=1) : {len(burglary_sampled):,}')
print(f'  Normal   (label=0) : {len(normal_sampled):,}')
print(f'  Total              : {len(burglary_sampled) + len(normal_sampled):,}')

---
## Cell 7 — Load Images into RAM

Reading directly from folders — much simpler than reading from ZIP!

⏱️ This takes **10–20 minutes**. Wait for it to finish.

In [ ]:
X = []   # Images
y = []   # Labels

start_time = time.time()

# --- Load Burglary images → label 1 ---
print(f'Loading {len(burglary_sampled):,} Burglary images...')
for idx, file_path in enumerate(burglary_sampled):
    img = Image.open(file_path).convert('RGB')  # Open directly from folder path
    X.append(np.array(img, dtype=np.uint8))     # Store at original 64x64 size
    y.append(1)                                 # Label 1 = Burglary
    if (idx + 1) % 5000 == 0:
        print(f'  -> {idx+1:,}/{len(burglary_sampled):,} loaded...')

# --- Load Normal images → label 0 ---
print(f'\nLoading {len(normal_sampled):,} Normal images...')
for idx, file_path in enumerate(normal_sampled):
    img = Image.open(file_path).convert('RGB')
    X.append(np.array(img, dtype=np.uint8))
    y.append(0)                                 # Label 0 = Normal
    if (idx + 1) % 5000 == 0:
        print(f'  -> {idx+1:,}/{len(normal_sampled):,} loaded...')

X = np.array(X, dtype=np.uint8)
y = np.array(y, dtype=np.int32)

elapsed = time.time() - start_time
print(f'\n✅ All images loaded in {elapsed/60:.1f} minutes!')
print(f'   X shape : {X.shape}')
print(f'   RAM used: ~{X.nbytes/1e9:.2f} GB')

---
## Cell 8 — Shuffle and Train/Validation Split (80/20)

In [ ]:
# Shuffle
indices = np.arange(len(X))
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

# 80/20 split
total      = len(X)
val_size   = int(0.2 * total)
train_size = total - val_size

X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = y[:train_size], y[train_size:]

print(f'✅ Split complete:')
print(f'   Training   : {train_size:,} images')
print(f'   Validation : {val_size:,} images')

---
## Cell 9 — Data Augmentation using ImageDataGenerator

- **Train**: horizontal flip + brightness (for CCTV variation)
- **Validation**: only resize + normalize (no augmentation)

Images are resized from 64x64 → 224x224 via `preprocessing_function`.

In [ ]:
def preprocess_image(img):
    """Resize to 224x224 and normalize pixel values to 0.0-1.0"""
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0
    return img.numpy()

# Train generator: WITH augmentation
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    preprocessing_function=preprocess_image
)

# Val generator: NO augmentation
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image
)

train_gen = train_datagen.flow(X_train, y_train, batch_size=32, shuffle=True)
val_gen   = val_datagen.flow(  X_val,   y_val,   batch_size=32, shuffle=False)

print('✅ Data generators ready!')
print(f'   Train batches : {len(train_gen)}')
print(f'   Val batches   : {len(val_gen)}')

---
## Cell 10 — Build EfficientNet-B0 Model

In [ ]:
print('Building EfficientNet-B0 model...')

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = True

inputs  = Input(shape=(224, 224, 3))
x       = base_model(inputs, training=True)
x       = GlobalAveragePooling2D()(x)
x       = Dropout(0.2)(x)
outputs = Dense(2, activation='softmax')(x)

model = Model(inputs, outputs)
print('✅ Model built!')
model.summary()

---
## Cell 11 — Compile

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print('✅ Model compiled!')

---
## Cell 12 — Define Callbacks

In [ ]:
# /kaggle/working/ is the output folder — files here can be downloaded
SAVE_PATH = '/kaggle/working/action_classifier.keras'

callbacks = [
    ModelCheckpoint(
        filepath=SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    ),
]
print('✅ Callbacks ready!')

---
## Cell 13 — Train the Model

⏱️ **Expected time on Kaggle GPU: ~30–60 minutes**

In [ ]:
print('Starting training...')
print('=' * 60)

history = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks
)

best_acc = max(history.history['val_accuracy']) * 100
print(f'\n✅ Training Complete!')
print(f'   Best Validation Accuracy : {best_acc:.2f}%')
print(f'   Model saved to           : {SAVE_PATH}')

---
## Cell 14 — Plot Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Accuracy', color='blue')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy',   color='orange')
ax1.set_title('Accuracy per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train Loss', color='blue')
ax2.plot(history.history['val_loss'], label='Val Loss',   color='orange')
ax2.set_title('Loss per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_results.png')  # Saved to output tab
plt.show()
print('✅ Training plot saved to output tab!')

---
## Cell 15 — Download Instructions

Your model is ready to download!

In [ ]:
import os

model_size = os.path.getsize(SAVE_PATH) / 1e6

print('=' * 55)
print('           TRAINING COMPLETE!')
print('=' * 55)
print(f'Model file : action_classifier.keras')
print(f'Size       : {model_size:.1f} MB')
print(f'Location   : {SAVE_PATH}')
print()
print('HOW TO DOWNLOAD:')
print('  1. Click the Output tab (right sidebar on Kaggle)')
print('  2. Find action_classifier.keras')
print('  3. Click the download arrow next to it')
print()
print('AFTER DOWNLOAD:')
print('  Place the file in: d:\\HOME-cctv\\action_classifier.keras')
print('  Run: python app.py')
print('  Your system will automatically load the Keras model!')
print('=' * 55)